# Etapa 4

In [24]:
# criando seção spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("INPE-MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

In [25]:
df = spark.read.parquet(
    "../data/silver/"
)

Antes de qualquer coisa do pipeline, vamos fazer uma breve análise das variáveis das nossas hipóteses agora

## Testando hipóteses

### Hipótese principal:

Menor precipitação/maior período sem chuva e menor umidade aumentam a probabilidade de FOCO_INTENSO = 1.

Analisando as possíveis variáveis temos seca_prolongada, baixa_umidade, condicao_seca e podemos criar mais uma

In [5]:
from pyspark.sql import functions as F

df.groupBy("seca_prolongada").agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).show()

+---------------+-------+------------------+
|seca_prolongada|      n| taxa_foco_intenso|
+---------------+-------+------------------+
|              1|1130038|0.4923064534113012|
|              0|3399157| 0.497963171456923|
+---------------+-------+------------------+



In [18]:
df.groupBy("baixa_umidade").agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).show()

+-------------+-------+------------------+
|baixa_umidade|      n| taxa_foco_intenso|
+-------------+-------+------------------+
|            1|1215253|0.6065123887783038|
|            0|3313942|0.4562282622930637|
+-------------+-------+------------------+



In [19]:
df.groupBy("condicao_seca").agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).show()

+-------------+-------+-------------------+
|condicao_seca|      n|  taxa_foco_intenso|
+-------------+-------+-------------------+
|            1| 406508| 0.5852799944896533|
|            0|4122687|0.48780297897948593|
+-------------+-------+-------------------+



Vamos testar também a precipitação acumulada que acabamos esquecendo, para isso vamos testar uma faixa de precipitação acumulada no dia:

In [20]:
df = df.withColumn(
    "faixa_precipitacao",
    F.when(F.col("precipitacao_inpe_mm") == 0, "0 mm")
     .when(F.col("precipitacao_inpe_mm") <= 1, "0-1 mm")
     .when(F.col("precipitacao_inpe_mm") <= 5, "1-5 mm")
     .otherwise(">5 mm")
)

In [22]:
df.groupBy("faixa_precipitacao").agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).orderBy("faixa_precipitacao").show()

+------------------+-------+------------------+
|faixa_precipitacao|      n| taxa_foco_intenso|
+------------------+-------+------------------+
|              0 mm|3971998|0.5024332842060847|
|            0-1 mm| 313455|0.4586240449187284|
|            1-5 mm| 158826|0.4568521526702177|
|             >5 mm|  84916|0.4357011635027557|
+------------------+-------+------------------+



In [23]:
df.groupBy("FOCO_INTENSO").agg(
    F.count("*").alias("n"),
    
    F.avg("umidade_relativa_ar_pct").alias(
        "umidade_media"
    ),
    
    F.avg("dia_sem_chuva").alias(
        "dias_sem_chuva_medio"
    ),
    
    F.avg("precipitacao_inpe_mm").alias(
        "precipitacao_media"
    )
).show()

+------------+-------+------------------+--------------------+-------------------+
|FOCO_INTENSO|      n|     umidade_media|dias_sem_chuva_medio| precipitacao_media|
+------------+-------+------------------+--------------------+-------------------+
|           1|2248980| 36.46994103993811|   32.93028839740683|0.32000678974469066|
|           0|2280215|41.624881864210174|   33.41522663433054| 0.3995076078352584|
+------------+-------+------------------+--------------------+-------------------+



A análise descritiva inicial apresentou indícios favoráveis à hipótese principal principalmente em relação à precipitação e à umidade. A proporção de focos intensos diminuiu conforme aumentaram os níveis de precipitação, passando de aproximadamente 50,24% nos registros sem precipitação para 43,57% nos registros acima de 5 mm. Da mesma forma, os focos classificados como intensos apresentaram umidade média menor (36,47%) do que os não intensos (41,62%). Por outro lado, dia_sem_chuva não apresentou uma diferença relevante entre as classes, com médias de 32,93 e 33,42 dias. Dessa forma, os resultados iniciais dão suporte principalmente à relação entre baixa precipitação, baixa umidade e focos mais intensos, enquanto o efeito da duração do período sem chuva ainda não apresenta evidências claras.

### Hipótese secundária 1:



Áreas com baixa precipitação e baixa umidade apresentam maior concentração espacial de focos com FRP acima da mediana (target = 1).

Espera-se identificar maior proporção de target = 1 nessas áreas e autocorrelação espacial positiva e significativa pelo índice de Moran Global (I > 0; p < 0,05) e Moran Local/LISA.

Com a análise que fizemos anteriormente, vemos que a condição seca está presente em 58% dos casos de foco intenso, mostrando que há relevância, porém vamos analisar mais a frente com o modelo se essas variáveis fazem sentido propriamente falando e depois fazer uma análise espacial.

### Hipótese secundária 2:

Maiores velocidades do vento estão associadas a uma maior probabilidade de ocorrência de focos com FRP acima da mediana (target = 1), principalmente em condições de baixa umidade e baixa precipitação antecedente.

A hipótese será rejeitada caso o vento não apresente associação positiva com a probabilidade de target = 1 ou caso essa associação não seja intensificada em condições mais secas.


Podemos fazer uma análise preliminar da variável vento e confirmar depois com o modelo:

In [6]:
df_vento = df.filter(
    F.col("vento_velocidade_horaria_ms").isNotNull()
)

print("Registros com vento:", df_vento.count())

Registros com vento: 509685


In [7]:
df_vento.groupBy("FOCO_INTENSO").agg(
    F.count("*").alias("n"),
    F.avg("vento_velocidade_horaria_ms").alias("vento_medio"),
    F.expr(
        "percentile_approx(vento_velocidade_horaria_ms, 0.5)"
    ).alias("vento_mediano")
).show()

+------------+------+-----------------+-------------+
|FOCO_INTENSO|     n|      vento_medio|vento_mediano|
+------------+------+-----------------+-------------+
|           1|247005|1.550782372826461|          1.0|
|           0|262680|1.211257042789706|          0.0|
+------------+------+-----------------+-------------+



In [8]:
q25, q50, q75 = df_vento.approxQuantile(
    "vento_velocidade_horaria_ms",
    [0.25, 0.50, 0.75],
    0.001
)

df_vento = df_vento.withColumn(
    "faixa_vento",
    F.when(F.col("vento_velocidade_horaria_ms") <= q25, "Q1")
     .when(F.col("vento_velocidade_horaria_ms") <= q50, "Q2")
     .when(F.col("vento_velocidade_horaria_ms") <= q75, "Q3")
     .otherwise("Q4")
)

In [9]:
df_vento.groupBy("faixa_vento").agg(
    F.count("*").alias("n"),
    F.avg("vento_velocidade_horaria_ms").alias("vento_medio"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).orderBy("faixa_vento").show()

+-----------+------+-----------------+-------------------+
|faixa_vento|     n|      vento_medio|  taxa_foco_intenso|
+-----------+------+-----------------+-------------------+
|         Q1|249023|              0.0| 0.4308678314854451|
|         Q2| 45197|              1.0|0.49956855543509526|
|         Q3|153238|2.465237082185881| 0.5369686370221486|
|         Q4| 62227|4.471676281999775| 0.5599820013820367|
+-----------+------+-----------------+-------------------+



In [10]:
df_vento.groupBy(
    "condicao_seca",
    "faixa_vento"
).agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).orderBy(
    "condicao_seca",
    "faixa_vento"
).show()

+-------------+-----------+------+-------------------+
|condicao_seca|faixa_vento|     n|  taxa_foco_intenso|
+-------------+-----------+------+-------------------+
|            0|         Q1|230176|0.41546468789100516|
|            0|         Q2| 40351|0.49133850462194245|
|            0|         Q3|130931| 0.5325553153951318|
|            0|         Q4| 52961|  0.556145087894866|
|            1|         Q1| 18847|  0.618984453759219|
|            1|         Q2|  4846| 0.5680973999174577|
|            1|         Q3| 22307| 0.5628726408750616|
|            1|         Q4|  9266| 0.5819123677962443|
+-------------+-----------+------+-------------------+



Na análise descritiva da segunda hipótese secundária, foi observado um aumento progressivo na proporção de focos intensos conforme aumentou a velocidade do vento. A taxa de FOCO_INTENSO = 1 passou de aproximadamente 43,09% no primeiro quartil de velocidade do vento para 56,00% no quarto quartil, indicando inicialmente uma associação positiva entre vento e intensidade dos focos. Entretanto, ao analisar essa relação separadamente de acordo com a condição de seca, o comportamento crescente não se manteve entre os registros classificados como secos. Dessa forma, os resultados preliminares dão suporte à associação entre maiores velocidades do vento e focos intensos, mas não mostram, até o momento, que esse efeito seja intensificado em condições mais secas. Essa relação será avaliada novamente durante a modelagem.

4A — Preparação para o MLlib
- Converter colunas categóricas com StringIndexer + OneHotEncoder
- Montar o vetor de features com VectorAssembler
- Dividir os dados em treino (70%) e teste (30%) usando randomSplit()
- Documentar a contagem de registros em cada split

Primeiramente vamos converter as colunas categóricas:

In [11]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

categoricas = [
    "periodo_dia",
    "bioma",
    "estado",
    "satelite"
]

indexers = [
    StringIndexer(
        inputCol=coluna,
        outputCol=f"{coluna}_idx",
        handleInvalid="keep"
    )
    for coluna in categoricas
]

encoders = [
    OneHotEncoder(
        inputCol=f"{coluna}_idx",
        outputCol=f"{coluna}_ohe",
        handleInvalid="keep"
    )
    for coluna in categoricas
]

Agora vamos montar o vetor de features

In [13]:
from pyspark.ml.feature import VectorAssembler

numericas = [
    "dia_sem_chuva",
    "precipitacao_inpe_mm",
    "umidade_relativa_ar_pct",
    "seca_prolongada",
    "baixa_umidade",
    "condicao_seca",
    "mes",
    "distancia_estacao_km",
    "dia_sem_chuva_missing",
    "precipitacao_missing",
    "umidade_missing"
]

assembler = VectorAssembler(
    inputCols=numericas + [
        "periodo_dia_ohe",
        "bioma_ohe",
        "estado_ohe",
        "satelite_ohe"
    ],
    outputCol="features"
)

Agora vamos dividir os dados em treino (70%) e teste (30%) usando randomSplit()

In [14]:
train, test = df.randomSplit(
    [0.7, 0.3],
    seed=42
)

In [15]:
df.count()

4529195

In [16]:
qtd_train = train.count()
qtd_train

3171055

In [17]:
qtd_test = test.count()
qtd_test

1358140

Tivemos 3171055 registros no treino e 1358140 registros no teste, de um total de 4529195 registros

4B — Construção do Pipeline
- Montar um Pipeline encadeando os estágios de preparação e o modelo, na ordem correta:
StringIndexer(s) → OneHotEncoder(s) → VectorAssembler → Modelo
- Treinar pelo menos 2 modelos diferentes do MLlib:
LogisticRegression e RandomForestClassifier

Vamos construir o pipeline agora dos nossos modelos:

In [18]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

lr = LogisticRegression(
    featuresCol="features",
    labelCol="FOCO_INTENSO"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="FOCO_INTENSO",
    seed=42
)

pipeline_lr = Pipeline(
    stages=
        indexers +
        encoders +
        [assembler, lr]
)

pipeline_rf = Pipeline(
    stages=
        indexers +
        encoders +
        [assembler, rf]
)

Gerando previsões dos modelos:

In [19]:
modelo_lr = pipeline_lr.fit(train)
modelo_rf = pipeline_rf.fit(train)

pred_lr = modelo_lr.transform(test)
pred_rf = modelo_rf.transform(test)

Agora criando os avaliadores:

In [20]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

evaluator_auc = BinaryClassificationEvaluator(
    labelCol="FOCO_INTENSO",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="FOCO_INTENSO",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="FOCO_INTENSO",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="FOCO_INTENSO",
    predictionCol="prediction",
    metricName="weightedRecall"
)

Avaliando a LogisticRegression:

In [21]:
auc_lr = evaluator_auc.evaluate(pred_lr)
acc_lr = evaluator_acc.evaluate(pred_lr)
precision_lr = evaluator_precision.evaluate(pred_lr)
recall_lr = evaluator_recall.evaluate(pred_lr)

print("Logistic Regression")
print(f"AUC-ROC: {auc_lr:.4f}")
print(f"Accuracy: {acc_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall: {recall_lr:.4f}")

Logistic Regression
AUC-ROC: 0.8482
Accuracy: 0.7627
Precision: 0.7951
Recall: 0.7627


E random forest:

In [22]:
auc_rf = evaluator_auc.evaluate(pred_rf)
acc_rf = evaluator_acc.evaluate(pred_rf)
precision_rf = evaluator_precision.evaluate(pred_rf)
recall_rf = evaluator_recall.evaluate(pred_rf)

print("Random Forest")
print(f"AUC-ROC: {auc_rf:.4f}")
print(f"Accuracy: {acc_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall: {recall_rf:.4f}")

Random Forest
AUC-ROC: 0.8407
Accuracy: 0.7504
Precision: 0.8093
Recall: 0.7504


E comparando ambos os resultados em uma tabela:

In [28]:
print(f"{'Modelo':<22} {'AUC-ROC':<10} {'Accuracy':<10} {'Precision':<10} {'Recall':<10}")

print(
    f"{'Logistic Regression':<22} "
    f"{auc_lr:<10.4f} "
    f"{acc_lr:<10.4f} "
    f"{precision_lr:<10.4f} "
    f"{recall_lr:<10.4f}"
)

print(
    f"{'Random Forest':<22} "
    f"{auc_rf:<10.4f} "
    f"{acc_rf:<10.4f} "
    f"{precision_rf:<10.4f} "
    f"{recall_rf:<10.4f}"
)

Modelo                 AUC-ROC    Accuracy   Precision  Recall    
Logistic Regression    0.8482     0.7627     0.7951     0.7627    
Random Forest          0.8407     0.7504     0.8093     0.7504    


Vamos testar também recall e precision por classe para verificar melhor se o modelo consegue captar bem os focos intensos:

In [29]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Precision da classe 1
evaluator_precision_1 = MulticlassClassificationEvaluator(
    labelCol="FOCO_INTENSO",
    predictionCol="prediction",
    metricName="precisionByLabel",
    metricLabel=1.0
)

# Recall da classe 1
evaluator_recall_1 = MulticlassClassificationEvaluator(
    labelCol="FOCO_INTENSO",
    predictionCol="prediction",
    metricName="recallByLabel",
    metricLabel=1.0
)

In [30]:
# Logistic Regression
precision_1_lr = evaluator_precision_1.evaluate(pred_lr)
recall_1_lr = evaluator_recall_1.evaluate(pred_lr)

# Random Forest
precision_1_rf = evaluator_precision_1.evaluate(pred_rf)
recall_1_rf = evaluator_recall_1.evaluate(pred_rf)

print("Logistic Regression")
print(f"Precision classe 1: {precision_1_lr:.4f}")
print(f"Recall classe 1:    {recall_1_lr:.4f}")

print("\nRandom Forest")
print(f"Precision classe 1: {precision_1_rf:.4f}")
print(f"Recall classe 1:    {recall_1_rf:.4f}")

Logistic Regression
Precision classe 1: 0.8928
Recall classe 1:    0.5930

Random Forest
Precision classe 1: 0.9449
Recall classe 1:    0.5276


Os dois modelos tiveram resultados relativamente próximos, mas a Regressão Logística apresentou um desempenho geral um pouco melhor, com AUC-ROC de 0,8482 e acurácia de 76,27%, enquanto a Random Forest apresentou 0,8407 e 75,04%, respectivamente. A Random Forest teve uma precisão geral um pouco maior, de 80,93% contra 79,51% da Regressão Logística.

Como nesse problema estamos mais interessados em conseguir identificar os focos realmente intensos, também olhamos especificamente para a classe FOCO_INTENSO = 1. Nesse caso, a Regressão Logística teve Recall de 59,30%, contra 52,76% da Random Forest. Ou seja, apesar dos dois modelos ainda deixarem passar uma quantidade considerável de focos intensos, a Regressão Logística consegue identificar uma parcela maior deles. Já a Random Forest apresentou uma Precision maior para essa classe, com 94,49% contra 89,28%.

Nesse contexto, acredito que o Recall seja uma métrica mais importante, pois um falso negativo representa um foco realmente intenso que o modelo não conseguiu identificar. Por isso, entre os dois modelos, a Regressão Logística parece ser a escolha mais adequada, tanto pelo maior Recall da classe positiva quanto pelos melhores resultados de AUC-ROC e acurácia. Mesmo assim, o Recall de 59,30% ainda pode ser considerado baixo para esse objetivo, já que aproximadamente 40% dos focos intensos continuam não sendo identificados. Por isso, uma possibilidade é testar diferentes thresholds de classificação para tentar aumentar o Recall, mesmo que isso resulte em uma redução da Precision.

In [32]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

pred_lr_threshold = pred_lr.withColumn(
    "prob_classe_1",
    vector_to_array("probability")[1]
)

In [33]:
pred_lr_threshold.select(
    "FOCO_INTENSO",
    "prob_classe_1",
    "prediction"
).show(10)

+------------+--------------------+----------+
|FOCO_INTENSO|       prob_classe_1|prediction|
+------------+--------------------+----------+
|           0|  0.0170485146775744|       0.0|
|           0| 0.03517307920973156|       0.0|
|           1|0.035161798631182295|       0.0|
|           0|0.022604910308945403|       0.0|
|           0| 0.03516295678560277|       0.0|
|           0|0.022937138639182786|       0.0|
|           0| 0.03517117590549257|       0.0|
|           0|0.029402545297119387|       0.0|
|           0|0.024254144562811497|       0.0|
|           0| 0.03716266265070367|       0.0|
+------------+--------------------+----------+
only showing top 10 rows


In [31]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]

In [34]:
resultados_threshold = []

for threshold in thresholds:

    pred_temp = pred_lr_threshold.withColumn(
        "prediction_threshold",
        F.when(
            F.col("prob_classe_1") >= threshold,
            1.0
        ).otherwise(0.0)
    )

    precision = MulticlassClassificationEvaluator(
        labelCol="FOCO_INTENSO",
        predictionCol="prediction_threshold",
        metricName="precisionByLabel",
        metricLabel=1.0
    ).evaluate(pred_temp)

    recall = MulticlassClassificationEvaluator(
        labelCol="FOCO_INTENSO",
        predictionCol="prediction_threshold",
        metricName="recallByLabel",
        metricLabel=1.0
    ).evaluate(pred_temp)

    accuracy = MulticlassClassificationEvaluator(
        labelCol="FOCO_INTENSO",
        predictionCol="prediction_threshold",
        metricName="accuracy"
    ).evaluate(pred_temp)

    resultados_threshold.append(
        (threshold, precision, recall, accuracy)
    )

In [35]:
print(
    f"{'Threshold':<12}"
    f"{'Precision':<12}"
    f"{'Recall':<12}"
    f"{'Accuracy':<12}"
)

for threshold, precision, recall, accuracy in resultados_threshold:
    print(
        f"{threshold:<12.2f}"
        f"{precision:<12.4f}"
        f"{recall:<12.4f}"
        f"{accuracy:<12.4f}"
    )

Threshold   Precision   Recall      Accuracy    
0.50        0.8928      0.5930      0.7627      
0.45        0.8136      0.6592      0.7560      
0.40        0.6952      0.7901      0.7240      
0.35        0.6217      0.9288      0.6843      
0.30        0.6016      0.9665      0.6658      
0.25        0.5941      0.9760      0.6572      


Como o Recall da Regressão Logística utilizando o threshold padrão de 0,50 foi de apenas 59,30%, também foram testados thresholds menores, buscando aumentar a capacidade do modelo de identificar focos realmente intensos. Como esperado, a redução do threshold aumentou o Recall, mas também provocou uma queda na Precision e na acurácia.

Entre os valores testados, o threshold de 0,40 apresentou um equilíbrio interessante para o objetivo do problema, aumentando o Recall de 59,30% para 79,01%, enquanto a Precision permaneceu em 69,52% e a acurácia em 72,40%. Dessa forma, o modelo passa a identificar aproximadamente 8 em cada 10 focos intensos, ao custo de um aumento nos falsos positivos. Como neste contexto consideramos mais prejudicial deixar de identificar um foco realmente intenso, essa troca pode ser considerada aceitável.

Nesse caso, fica como análise posterior essa questão já que teriamos que dividir novamente entre treino e teste para ser metodologicamente mais aceitável

4D — Importância das Features
- Para modelos baseados em árvore: extrair e visualizar os feature importances
( .featureImportances )
- Para Regressão Linear / Logística: extrair e interpretar os coeficientes ( .coefficients )
- Listar as Top 5 features mais importantes e explicar se fazem sentido à luz do domínio do
problema

Vamos seguir com o modelo de regressão linear e recuperar os coeficientes:

In [38]:
lr_model = modelo_lr.stages[-1]

print(lr_model)
print(lr_model.coefficients)

LogisticRegressionModel: uid=LogisticRegression_ce7257633c08, numClasses=2, numFeatures=41
[0.0012741282418198452,-0.0062776731231689075,-0.002969807528077063,-0.0038997925797682135,0.1321083783891026,-0.004343610424049379,0.06672682345024873,0.0010057182328772033,-0.11065681114776758,-0.11698994146937618,0.058551972923834204,0.8050601617783109,-1.7587473535123677,1.1142372750206893,-0.06522130545423646,0.0,0.16110757370656403,-0.15914612733575326,0.19312766428550135,-0.5789037468332532,0.0,-0.20315517143130574,-0.027703966085995752,0.2827382927969288,0.1215862209101772,-0.07181225997005848,-0.043434702272935145,-0.005582223281743688,0.3653587698512249,0.03007116768132254,0.0,-5.718159272475738,-5.715116054431625,22.36428119398205,-5.828042578624379,-5.377871422877699,-2.676093458051975,-3.3493380884057116,-2.00918526335473,-2.1391608948352046,0.0]


Temos os coeficientes sem os nomes, vamos recuperar dos metadados do vectorassembler

In [39]:
attrs = pred_lr.schema["features"].metadata["ml_attr"]["attrs"]

features_info = []

for tipo in attrs:
    for attr in attrs[tipo]:
        features_info.append(
            (attr["idx"], attr["name"])
        )

features_info = sorted(features_info)

In [40]:
coeficientes = lr_model.coefficients.toArray()

resultado_coef = [
    (nome, float(coeficientes[idx]))
    for idx, nome in features_info
]

In [45]:
resultado_coef_ordenado = sorted(
    resultado_coef,
    key=lambda x: abs(x[1]),
    reverse=True
)

print(f"{'Feature':<50} {'Coeficiente':>12}")

for feature, coef in resultado_coef_ordenado[:50]:
    print(f"{feature:<50} {coef:>12.4f}")

Feature                                             Coeficiente
satelite_ohe_GOES-16                                    22.3643
satelite_ohe_NPP-375D                                   -5.8280
satelite_ohe_NOAA-20                                    -5.7182
satelite_ohe_NPP-375                                    -5.7151
satelite_ohe_NOAA-21                                    -5.3779
satelite_ohe_TERRA_M-T                                  -3.3493
satelite_ohe_AQUA_M-T                                   -2.6761
satelite_ohe_AQUA_M-M                                   -2.1392
satelite_ohe_TERRA_M-M                                  -2.0092
periodo_dia_ohe_madrugada                               -1.7587
periodo_dia_ohe_noite                                    1.1142
periodo_dia_ohe_tarde                                    0.8051
bioma_ohe_Mata Atlântica                                -0.5789
estado_ohe_ALAGOAS                                       0.3654
estado_ohe_BAHIA                        

In [46]:
top5_lr = resultado_coef_ordenado[:5]

print("Top 5 - Logistic Regression\n")

for i, (feature, coef) in enumerate(top5_lr, 1):
    direcao = "positiva" if coef > 0 else "negativa"

    print(
        f"{i}. {feature}: "
        f"{coef:.4f} ({direcao})"
    )

Top 5 - Logistic Regression

1. satelite_ohe_GOES-16: 22.3643 (positiva)
2. satelite_ohe_NPP-375D: -5.8280 (negativa)
3. satelite_ohe_NOAA-20: -5.7182 (negativa)
4. satelite_ohe_NPP-375: -5.7151 (negativa)
5. satelite_ohe_NOAA-21: -5.3779 (negativa)


Aqui o mais interessante é ver que o top 5 é dominado pelos satelites, antes de qualquer conclusão, vamos investigar a distribuição do foco intenso pelos satelites:

In [47]:
from pyspark.sql import functions as F

df.groupBy("satelite").agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso"),
    F.avg("frp").alias("frp_medio"),
    F.expr("percentile_approx(frp, 0.5)").alias("frp_mediano")
).orderBy(
    F.desc("taxa_foco_intenso")
).show(truncate=False)

+---------+-------+-------------------+------------------+-----------+
|satelite |n      |taxa_foco_intenso  |frp_medio         |frp_mediano|
+---------+-------+-------------------+------------------+-----------+
|GOES-16  |829736 |1.0                |119.84283073170269|93.9       |
|AQUA_M-T |239461 |0.9305565415662675 |60.53470460743107 |29.2       |
|TERRA_M-T|100030 |0.8661201639508147 |41.322188343496855|21.6       |
|TERRA_M-M|67397  |0.6565277386233809 |28.171262815852295|13.7       |
|AQUA_M-M |20998  |0.6314887132107819 |26.591437279740926|12.6       |
|NPP-375  |1231615|0.3998976953025093 |15.128575650670033|7.8        |
|NOAA-21  |294812 |0.3625870045995414 |13.534671926515932|7.0        |
|NOAA-20  |1383734|0.3160947118449066 |12.31540809143945 |6.2        |
|NPP-375D |361412 |0.04277666485894215|3.10539910130268  |1.8        |
+---------+-------+-------------------+------------------+-----------+



Analisando os coeficientes da Regressão Logística, percebemos que as variáveis relacionadas ao satélite acabaram tendo os maiores pesos no modelo. Quando fomos olhar melhor essa relação, vimos que existe uma diferença muito grande na proporção de focos intensos entre os satélites. No caso do GOES-16, por exemplo, todos os 829.736 registros foram classificados como FOCO_INTENSO = 1, enquanto no NPP-375D apenas cerca de 4,28% ficaram na classe positiva. Também existe uma diferença grande no FRP mediano entre os sensores, indo de 1,8 no NPP-375D até 93,9 no GOES-16. Isso ajuda a explicar por que a variável satelite teve tanta influência no modelo. Mesmo assim, é importante ter cuidado com essa interpretação, já que essa diferença pode estar mais relacionada às características dos sensores e da forma de detecção do que às condições ambientais que estamos tentando analisar.

Vamos testar rapidamente um modelo sem satelite para ver o que acontece

In [49]:
categoricas_sem_satelite = [
    "periodo_dia",
    "bioma",
    "estado"
]

indexers_sem_satelite = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categoricas_sem_satelite
]

encoders_sem_satelite = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe",
        handleInvalid="keep"
    )
    for c in categoricas_sem_satelite
]

In [50]:
assembler_sem_satelite = VectorAssembler(
    inputCols=numericas + [
        "periodo_dia_ohe",
        "bioma_ohe",
        "estado_ohe"
    ],
    outputCol="features"
)

In [51]:
pipeline_lr_sem_satelite = Pipeline(
    stages=
        indexers_sem_satelite +
        encoders_sem_satelite +
        [assembler_sem_satelite, lr]
)

In [52]:
modelo_lr_sem_satelite = pipeline_lr_sem_satelite.fit(train)

pred_lr_sem_satelite = modelo_lr_sem_satelite.transform(test)

In [53]:
auc_lr_sem_sat = evaluator_auc.evaluate(pred_lr_sem_satelite)
acc_lr_sem_sat = evaluator_acc.evaluate(pred_lr_sem_satelite)

precision_1_lr_sem_sat = evaluator_precision_1.evaluate(
    pred_lr_sem_satelite
)

recall_1_lr_sem_sat = evaluator_recall_1.evaluate(
    pred_lr_sem_satelite
)

print("Logistic Regression sem satélite")
print(f"AUC-ROC: {auc_lr_sem_sat:.4f}")
print(f"Accuracy: {acc_lr_sem_sat:.4f}")
print(f"Precision classe 1: {precision_1_lr_sem_sat:.4f}")
print(f"Recall classe 1: {recall_1_lr_sem_sat:.4f}")

Logistic Regression sem satélite
AUC-ROC: 0.7392
Accuracy: 0.6497
Precision classe 1: 0.6474
Recall classe 1: 0.6455


Olhando os coeficientes do modelo sem satelite:

In [54]:
lr_model_sem_sat = modelo_lr_sem_satelite.stages[-1]

coeficientes_sem_sat = lr_model_sem_sat.coefficients.toArray()

In [55]:
attrs = (
    pred_lr_sem_satelite
    .schema["features"]
    .metadata["ml_attr"]["attrs"]
)

features_info = []

for tipo in attrs:
    for attr in attrs[tipo]:
        features_info.append(
            (attr["idx"], attr["name"])
        )

features_info = sorted(features_info)

In [56]:
resultado_coef_sem_sat = [
    (nome, float(coeficientes_sem_sat[idx]))
    for idx, nome in features_info
]

In [57]:
resultado_coef_sem_sat = sorted(
    resultado_coef_sem_sat,
    key=lambda x: abs(x[1]),
    reverse=True
)

In [58]:
print(f"{'Feature':<50} {'Coeficiente':>12}")

for feature, coef in resultado_coef_sem_sat:
    print(f"{feature:<50} {coef:>12.4f}")

Feature                                             Coeficiente
periodo_dia_ohe_noite                                    6.9287
periodo_dia_ohe_manha                                    3.8329
periodo_dia_ohe_madrugada                               -2.7295
periodo_dia_ohe_tarde                                   -1.1870
estado_ohe_ALAGOAS                                       0.7259
bioma_ohe_Mata Atlântica                                -0.6652
estado_ohe_BAHIA                                         0.2486
bioma_ohe_Cerrado                                        0.2106
estado_ohe_MARANHÃO                                     -0.1846
bioma_ohe_Caatinga                                      -0.1841
dia_sem_chuva_missing                                   -0.1793
umidade_missing                                         -0.1608
estado_ohe_PERNAMBUCO                                   -0.1510
bioma_ohe_Amazônia                                       0.1503
estado_ohe_PARAÍBA                      

In [59]:
print("\nTop 5 - Logistic Regression sem satélite\n")

for i, (feature, coef) in enumerate(
    resultado_coef_sem_sat[:5],
    1
):
    direcao = "positiva" if coef > 0 else "negativa"

    print(
        f"{i}. {feature}: "
        f"{coef:.4f} ({direcao})"
    )


Top 5 - Logistic Regression sem satélite

1. periodo_dia_ohe_noite: 6.9287 (positiva)
2. periodo_dia_ohe_manha: 3.8329 (positiva)
3. periodo_dia_ohe_madrugada: -2.7295 (negativa)
4. periodo_dia_ohe_tarde: -1.1870 (negativa)
5. estado_ohe_ALAGOAS: 0.7259 (positiva)


Ao retirar a variável satelite, percebemos que o período do dia passou a ter uma influência bem maior no modelo. As quatro primeiras features com maiores coeficientes estão relacionadas ao período da ocorrência, com noite e manhã apresentando associação positiva com a classificação como foco intenso, enquanto madrugada e tarde apresentaram associação negativa. Além disso, o estado de Alagoas apareceu como a quinta feature com maior coeficiente. Esse resultado mostra que, mesmo retirando o efeito do satélite, características temporais e geográficas ainda possuem bastante influência na classificação dos focos.

Por último vamos criar um modelo que inclua o vento para conseguirmos verificar corretamente a resposta da hipótese secundária 2, para isso a ideia é pegar o modelo sem satélite:

Antes vamos criar uma feature que associa vento à condição seca (baixa_umidade + seca_prolongada) para verificar o comportamento dessas duas features juntas

In [60]:
from pyspark.sql import functions as F

df_vento = (
    df
    .filter(F.col("vento_velocidade_horaria_ms").isNotNull())
    .withColumn(
        "vento_condicao_seca",
        F.col("vento_velocidade_horaria_ms") * F.col("condicao_seca")
    )
)

In [61]:
categoricas_vento = [
    "periodo_dia",
    "bioma",
    "estado"
]

indexers_vento = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categoricas_vento
]

encoders_vento = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe",
        handleInvalid="keep"
    )
    for c in categoricas_vento
]

In [62]:
numericas_vento = [
    "dia_sem_chuva",
    "precipitacao_inpe_mm",
    "umidade_relativa_ar_pct",

    "seca_prolongada",
    "baixa_umidade",
    "condicao_seca",

    "vento_velocidade_horaria_ms",
    "vento_condicao_seca",

    "mes",
    "distancia_estacao_km",

    "dia_sem_chuva_missing",
    "precipitacao_missing",
    "umidade_missing"
]

In [63]:
assembler_vento = VectorAssembler(
    inputCols=numericas_vento + [
        "periodo_dia_ohe",
        "bioma_ohe",
        "estado_ohe"
    ],
    outputCol="features"
)

In [64]:
train_vento, test_vento = df_vento.randomSplit(
    [0.7, 0.3],
    seed=42
)

print("Treino:", train_vento.count())
print("Teste:", test_vento.count())

Treino: 356967
Teste: 152718


In [65]:
lr_vento = LogisticRegression(
    featuresCol="features",
    labelCol="FOCO_INTENSO"
)

pipeline_lr_vento = Pipeline(
    stages=
        indexers_vento +
        encoders_vento +
        [assembler_vento, lr_vento]
)

In [66]:
modelo_lr_vento = pipeline_lr_vento.fit(train_vento)

pred_lr_vento = modelo_lr_vento.transform(test_vento)

In [67]:
auc_vento = evaluator_auc.evaluate(pred_lr_vento)
acc_vento = evaluator_acc.evaluate(pred_lr_vento)

precision_vento = evaluator_precision_1.evaluate(
    pred_lr_vento
)

recall_vento = evaluator_recall_1.evaluate(
    pred_lr_vento
)

print("Logistic Regression com vento")
print(f"AUC-ROC: {auc_vento:.4f}")
print(f"Accuracy: {acc_vento:.4f}")
print(f"Precision classe 1: {precision_vento:.4f}")
print(f"Recall classe 1: {recall_vento:.4f}")

Logistic Regression com vento
AUC-ROC: 0.7870
Accuracy: 0.6989
Precision classe 1: 0.6761
Recall classe 1: 0.7323


In [68]:
lr_model_vento = modelo_lr_vento.stages[-1]

coeficientes_vento = lr_model_vento.coefficients.toArray()

attrs = (
    pred_lr_vento
    .schema["features"]
    .metadata["ml_attr"]["attrs"]
)

features_info = []

for tipo in attrs:
    for attr in attrs[tipo]:
        features_info.append(
            (attr["idx"], attr["name"])
        )

features_info = sorted(features_info)

resultado_coef_vento = [
    (nome, float(coeficientes_vento[idx]))
    for idx, nome in features_info
]

resultado_coef_vento = sorted(
    resultado_coef_vento,
    key=lambda x: abs(x[1]),
    reverse=True
)

for feature, coef in resultado_coef_vento:
    print(f"{feature:<50} {coef:>12.4f}")

periodo_dia_ohe_noite                                    6.9346
periodo_dia_ohe_manha                                    4.1789
periodo_dia_ohe_madrugada                               -2.5166
periodo_dia_ohe_tarde                                   -1.1032
estado_ohe_ALAGOAS                                       0.7526
bioma_ohe_Mata Atlântica                                -0.5339
estado_ohe_PARAÍBA                                      -0.3903
estado_ohe_PERNAMBUCO                                   -0.3837
dia_sem_chuva_missing                                   -0.3325
estado_ohe_BAHIA                                         0.2774
bioma_ohe_Amazônia                                       0.2340
estado_ohe_SERGIPE                                      -0.2318
umidade_missing                                          0.1496
seca_prolongada                                         -0.1433
baixa_umidade                                           -0.1260
bioma_ohe_Caatinga                      

No modelo complementar com os registros que possuem informação de vento, a variável vento_velocidade_horaria_ms apresentou coeficiente positivo de 0,0238, indicando que maiores velocidades do vento estão associadas a uma maior probabilidade de ocorrência de focos intensos. Além disso, a interação vento_condicao_seca também apresentou coeficiente positivo, de 0,0275, sugerindo que o efeito do vento tende a ser maior quando ocorre junto a condições mais secas. Esses resultados vão na direção esperada pela segunda hipótese secundária. Mesmo assim, essa conclusão deve ser interpretada com cautela, já que o modelo foi treinado apenas no subconjunto em que havia dados de vento disponíveis.

**Conclusão final sobre a etapa 4:**

## Análise das features mais importantes

Para entender quais variáveis tiveram maior influência na classificação dos focos intensos, foram analisados os coeficientes da Regressão Logística. Como as variáveis possuem escalas diferentes e algumas passaram por One-Hot Encoding, os valores dos coeficientes não devem ser interpretados diretamente como uma medida absoluta de importância. Mesmo assim, a análise dos maiores coeficientes em módulo permite entender quais features tiveram maior influência nas decisões do modelo.

### Modelo principal com satélite

No modelo principal, as cinco features com maiores coeficientes em valor absoluto foram:

| Posição | Feature | Coeficiente |
|---|---|---:|
| 1 | GOES-16 | +22,3643 |
| 2 | NPP-375D | -5,8280 |
| 3 | NOAA-20 | -5,7182 |
| 4 | NPP-375 | -5,7151 |
| 5 | NOAA-21 | -5,3779 |

As cinco features estão relacionadas ao satélite utilizado na detecção dos focos. O `GOES-16` apresentou um coeficiente positivo muito alto, enquanto `NPP-375D`, `NOAA-20`, `NPP-375` e `NOAA-21` apresentaram coeficientes negativos.

Investigando melhor esse resultado, percebemos que existe uma diferença muito grande na distribuição da target entre os satélites. No caso do GOES-16, todos os 829.736 registros disponíveis foram classificados como `FOCO_INTENSO = 1`, enquanto no NPP-375D apenas cerca de 4,28% dos registros ficaram na classe positiva. Também existe uma diferença grande no FRP mediano, indo de 1,8 no NPP-375D até 93,9 no GOES-16.

Isso ajuda a explicar por que o satélite teve tanta influência no modelo. Porém, é importante ter cuidado com essa interpretação, já que essa diferença pode estar relacionada às características dos sensores e à forma como os focos são detectados, e não diretamente às condições ambientais que estamos tentando analisar.

O modelo principal apresentou:

- **AUC-ROC:** 0,8482
- **Accuracy:** 76,27%
- **Precision da classe 1:** 89,28%
- **Recall da classe 1:** 59,30%

---

### Modelo complementar sem satélite

Como o satélite apresentou uma influência muito grande no primeiro modelo, também foi treinada uma Regressão Logística retirando essa variável. A ideia foi verificar como o modelo se comportaria utilizando principalmente as características ambientais, temporais e geográficas.

As cinco features com maiores coeficientes em valor absoluto foram:

| Posição | Feature | Coeficiente |
|---|---|---:|
| 1 | Período do dia - noite | +6,9287 |
| 2 | Período do dia - manhã | +3,8329 |
| 3 | Período do dia - madrugada | -2,7295 |
| 4 | Período do dia - tarde | -1,1870 |
| 5 | Estado - Alagoas | +0,7259 |

Sem a variável `satelite`, percebemos que o período do dia passou a ter uma influência bem maior no modelo. Noite e manhã apresentaram associação positiva com a classificação como foco intenso, enquanto madrugada e tarde apresentaram associação negativa. O estado de Alagoas também apareceu entre as cinco features com maiores coeficientes.

Esses resultados mostram que, retirando a influência do sensor, características temporais e geográficas passam a ter maior peso na classificação. Essas relações, porém, devem ser entendidas como associações encontradas pelo modelo e não necessariamente como relações de causa e efeito.

O desempenho do modelo sem satélite foi:

- **AUC-ROC:** 0,7392
- **Accuracy:** 64,97%
- **Precision da classe 1:** 64,74%
- **Recall da classe 1:** 64,55%

A queda da AUC-ROC de 0,8482 para 0,7392 e da acurácia de 76,27% para 64,97% mostra que a informação do satélite possui uma influência considerável no poder preditivo do modelo.

Apesar das variáveis meteorológicas não aparecerem entre as cinco primeiras, algumas apresentaram coeficientes na direção esperada pelas hipóteses, como a precipitação e a umidade, que apresentaram coeficientes negativos, enquanto a condição seca apresentou coeficiente positivo.

---

### Modelo complementar com vento

Por último, foi criado um modelo complementar utilizando apenas os registros que possuíam informação de velocidade do vento. Esse modelo foi utilizado principalmente para ajudar na análise da segunda hipótese secundária.

As cinco features com maiores coeficientes em valor absoluto foram:

| Posição | Feature | Coeficiente |
|---|---|---:|
| 1 | Período do dia - noite | +6,9346 |
| 2 | Período do dia - manhã | +4,1789 |
| 3 | Período do dia - madrugada | -2,5166 |
| 4 | Período do dia - tarde | -1,1032 |
| 5 | Estado - Alagoas | +0,7526 |

Novamente, as features relacionadas ao período do dia tiveram os maiores coeficientes, seguidas pelo estado de Alagoas.

Apesar das variáveis relacionadas ao vento não aparecerem entre as cinco primeiras, elas apresentaram resultados importantes para a hipótese:

| Feature | Coeficiente |
|---|---:|
| Velocidade do vento | +0,0238 |
| Vento × condição seca | +0,0275 |

A velocidade do vento apresentou coeficiente positivo, indicando que maiores velocidades estão associadas a uma maior probabilidade de ocorrência de `FOCO_INTENSO = 1`. Além disso, a interação entre vento e condição seca também apresentou coeficiente positivo, indicando que essa associação tende a ficar mais forte quando o vento ocorre junto a condições mais secas.

Esses resultados estão na direção esperada pela segunda hipótese secundária. Mesmo assim, essa análise deve ser vista como complementar, já que foi realizada somente com o subconjunto dos dados em que a informação de velocidade do vento estava disponível.

---

### Conclusão sobre as features mais importantes

De forma geral, percebemos que as features mais influentes para a classificação não foram necessariamente as variáveis meteorológicas relacionadas diretamente às hipóteses.

No modelo principal, o satélite teve uma influência muito forte, principalmente devido às grandes diferenças encontradas na distribuição do FRP e da target entre os diferentes sensores. Quando retiramos essa variável, o período do dia e algumas características geográficas passaram a apresentar os maiores coeficientes.

Mesmo não aparecendo entre as Top 5, variáveis como precipitação, umidade, condição seca e vento apresentaram relações importantes e, em alguns casos, na direção esperada pelas hipóteses. Por isso, essas variáveis serão analisadas em conjunto com os resultados descritivos e dos modelos para responder às hipóteses propostas no trabalho.

Vale destacar que os rankings apresentados representam os maiores coeficientes em valor absoluto da Regressão Logística. Como existem variáveis em escalas diferentes e variáveis categóricas transformadas por One-Hot Encoding, a magnitude dos coeficientes deve ser interpretada com cautela e não como uma medida absoluta de importância entre todas as features.
```


# Etapa 5

Não farei a etapa 5 devido a falta de tempo, estou terminando isso na manhã final do prazo (quinta - 13/08/2026)

# Etapa 6

1. MLlib vs. Scikit-learn: Você já usou (ou conhece) Scikit-learn para treinar modelos em datasets menores. Compare a experiência de usar MLlib. O que é mais difícil? O que é mais fácil? Em que cenários reais de negócio um cientista de dados precisa usar MLlib em vez de Scikit-learn? Dê um exemplo concreto.

A diferença principal aqui foi a de ter que ser mais verboso por conta da nomenclatura do pyspark, por ser uma integração baseada em java e python principalmente. Mesmo tendo essa camada de tradução que é o python para o spark, senti a necessidade de ser cada vez mais claro com declarar o que é cada coisa. Por exemplo, não me lembro de precisar usar o vector assembler explicitamente, senti que o scikit-learn fazia isso mais por debaixo dos panos. Achei a princípio o scikit-learn mais fácil. Porém, nem todos os casos é possível usar somente o pandas com scikit-learn justamente por causa da quantidade de dados, ainda mais hoje no escopo do big data. Por exemplo, nesse caso aqui mesmo de 250 milhões de linhas quase não consegui analisar com 8gb do meu computador, e isso se revela em casos de empresas porque elas trabalham com milhões e milhões de linhas. Na empresa que eu estagio atualmente só a parte de faturamento nos últimos 5 anos tem quase 10 milhões de linhas e isso acaba não cabendo na memória ram somente, então seria necessário pensar em usar spark mesmo que local para conseguir rodar isso e construir modelos.

2. Do Modelo à Decisão: Imagine que o gestor público (ou o diretor da empresa) pergunta: "O que este modelo significa na prática? Que decisão eu tomo a partir dele?" Responda a essa pergunta no contexto do seu domínio específico. Seja concreto — não dê respostas genéricas.

O modelo tem a função de capturar a maior probabilidade de ter um foco intenso ou não, e isso está relacionado a periodos do dia, condições meteorológicas, e principalmente aos satélites específicos (como visto pelos coeficientes no modelo de regressão logística). Algumas decisões que pensei foram, em relação ao periodo do dia, ter maior ou menor atenção dependendo do periodo (madrugada e de tarde tem menor probabilidade de ter foco intenso pelo coeficiente negativo), então deveria prestar mais atenção a lugares que tem condições de baixa_umidade (que apesar de não ter sido averiguado pelo modelo devido ao menor coeficiente, é confirmada pelo nosso teste de hipótese anterior descritivo, assim como lugares onde a precipitação media é menor). Uma que eu considero bastante interessante é, por exemplo, no caso do modelo GOES, ele teve 100% de foco intenso, então a atenção para apagar possíveis focos de incêndio poderia ser totalmente voltada incialmente para os lugares que tem maior proximidade com esse satélite, porque algo de incomum pode estar acontecendo lá nesse satélite, investigando melhor o que estaria acontecendo. Essa parte dos satélites merece atenção porque pode haver algumas diferenças entre a maneira como eles medem o FRP, o que pode acabar enviesando. Lugares que tem mais vento também tem a tendência de ter focos mais intensos, então decisões para chamar unidades que lidem com focos de incêndio poderiam ser voltadas para lugares nessas condições, começando pelos satélites por conta dos coeficientes e da sua representatividade nisso.

Eu diria que seriam duas tomadas de decisão aqui, com o modelo servindo como ferramenta pra priorizar certas decisões, ele poderia usar o modelo para gerar a probabilidade prevista junto as condições meteorológicas para definir o que deveria ser analisado primeiro. A outra coisa seria investigar melhor isso dos satélites, porque 100% de foco intenso em um satélite é muito característico de um viés. No geral acredito que seria de grande ajuda para decidir quais focos priorizar.

3. Viés e Limitações Todo modelo tem limitações. Identifique pelo menos 2 fontes de viés nos seus dados e explique como elas podem afetar as previsões do modelo. Exemplos: sub-representação geográfica, viés de registro (só aparece no dado quem passou pelo sistema formal), sazonalidade não capturada.

O primeiro aqui bem claro é o dos satélites, que provavelmente existe devido à diferença de medição entre eles ou algo incomum que possa estar acontecendo. Temos satélites como o GOES que tem 100% de foco intenso, enquanto o NPP tem 4,28%, o que sugere uma diferença na medição entre eles. Isso ficou bem evidente quanto tiramos a variável satélite por exemplo, até o AUC-ROC caiu de quase 85% para aproximadamente 75%, já que os satélites representam uma boa parte da explicação do modelo.

O segundo que poderia ser apontado é a questão da proximidade das estações meteorológicas, porque um foco localizado próximo de uma estação provavelmente tem uma medição meteorológica mais representativa do que um foco muito distante, e isso vem da variável de distancia que calculamos (distancia_estacao_km). Isso pode acabar gerando um viés geográfico, já que regiões com maior cobertura de estações pode ter maiores informações relacionadas ao medição meteorológica. Isso nos leva a pensar também nos estados, já que alguns estados podem ter maiores representações de focos do que outros considerando suas diferentes condições climáticas e cobertura tecnológica.